In [ ]:
import numpy as np

X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

# LSTM attend des séquences (samples, timesteps, features)
TIMESTEPS = 10
N_FEATURES = X_train.shape[1]

def make_sequences(X, y, timesteps):
    Xs, ys = [], []
    for i in range(len(X) - timesteps):
        Xs.append(X[i:i+timesteps])
        ys.append(y[i+timesteps])
    return np.array(Xs), np.array(ys)

# Sous-ensemble pour aller plus vite
X_tr_seq, y_tr_seq = make_sequences(X_train[:50000], y_train[:50000], TIMESTEPS)
X_te_seq, y_te_seq = make_sequences(X_test[:10000],  y_test[:10000],  TIMESTEPS)

print("X_train sequences :", X_tr_seq.shape)
print("X_test  sequences :", X_te_seq.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

N_CLASSES = len(np.unique(y_train))
y_tr_cat  = to_categorical(y_tr_seq, N_CLASSES)
y_te_cat  = to_categorical(y_te_seq, N_CLASSES)

model = Sequential([
    LSTM(64, input_shape=(TIMESTEPS, N_FEATURES), return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(N_CLASSES, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint('../models/lstm_model.h5', save_best_only=True)
]

history = model.fit(
    X_tr_seq, y_tr_cat,
    epochs=20,
    batch_size=256,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)
print("Modèle sauvegardé : models/lstm_model.h5")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'],     label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'],     label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title("Loss")
plt.legend()

plt.tight_layout()
plt.savefig('../data/lstm_training_curves.png')
plt.show()